In [6]:
#Start Spark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType, IntegerType


spark = (SparkSession.builder
        .appName("Spark Basics Assignment")
        .getOrCreate())

In [8]:
#Load Data
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", True)
    .csv("/content/Superstore.csv")
)
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [9]:
#First few rows
df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [21]:
#Columns
print(df.columns)
#data types schema
df.printSchema()

['Row ID', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'Segment', 'Country', 'City', 'State', 'postal_code', 'Region', 'product_id', 'Category', 'sub_category', 'product_name', 'Sales', 'Quantity', 'Discount', 'Profit']
root
 |-- Row ID: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- postal_code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Qua

In [11]:
#Remove duplicates
rows_before = df.count()
df = df.dropDuplicates()
rows_after = df.count()
print(f"rows before removing duplicates:{rows_before}")
print(f"rows after removing duplicates:{rows_after}")

rows before removing duplicates:9994
rows after removing duplicates:9994


In [12]:
#Check null values
count_null=df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
count_null.show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [13]:
rows_before_drop = df.count()
print(f"rows before :{rows_before_drop}")
df = df.na.drop()
rows_after_drop = df.count()
print(f"rows after :{rows_after_drop}")

rows before :9994
rows after :9994


In [23]:
#Check for inconsistent value
inconsistent_sales = df.filter(col("Sales") < 0).count()
inconsistent_discount = df.filter((col("Discount") < 0) | (col("Discount") > 1)).count()
print(f"Rows with negative Sales: {inconsistent_sales}")
print(f"Rows with Discount outside [0,1]: {inconsistent_discount}")

Rows with negative Sales: 0
Rows with Discount outside [0,1]: 0


In [14]:
# Filter by Region
west_df = df.filter(col("Region") == "West")
print(f"\nRows in West region: {west_df.count()}")

# Filter by Category
furniture_df = df.filter(col("Category") == "Furniture")
print(f"Rows in Furniture category: {furniture_df.count()}")

# Filter by a numeric condition
high_sales_df = df.filter(col("Sales") > 500)
print(f"Rows with Sales > 500: {high_sales_df.count()}")

# Combined filter
combo_df = df.filter((col("Region") == "West") & (col("Category") == "Furniture"))
print(f"Rows in West region AND Furniture category: {combo_df.count()}")



Rows in West region: 3203
Rows in Furniture category: 2121
Rows with Sales > 500: 1162
Rows in West region AND Furniture category: 707


In [15]:
df = (
    df.withColumnRenamed("Order ID", "order_id")
      .withColumnRenamed("Order Date", "order_date")
      .withColumnRenamed("Ship Date", "ship_date")
      .withColumnRenamed("Ship Mode", "ship_mode")
      .withColumnRenamed("Customer ID", "customer_id")
      .withColumnRenamed("Customer Name", "customer_name")
      .withColumnRenamed("Postal Code", "postal_code")
      .withColumnRenamed("Product ID", "product_id")
      .withColumnRenamed("Sub-Category", "sub_category")
      .withColumnRenamed("Product Name", "product_name")
)

# Change data types: postal_code
df = df.withColumn("postal_code", col("postal_code").cast(IntegerType()))

# Convert order_date string -> actual date type
df = df.withColumn("order_date", to_date("order_date", "M/d/yyyy"))
df = df.withColumn("ship_date", to_date("ship_date", "M/d/yyyy"))

print("\n Schema after transformation ")
df.printSchema()



 Schema after transformation 
root
 |-- Row ID: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- postal_code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [16]:
#Aggregration
print("\n Basic aggregations ")
df.select(count("*").alias("total_rows"),
    avg("Sales").alias("avg_sales"),
    min("Sales").alias("min_sales"),
    max("Sales").alias("max_sales"),
    avg("Profit").alias("avg_profit"),
    min("Profit").alias("min_profit"),
    max("Profit").alias("max_profit"),).show()



--- Basic aggregations ---
+----------+-----------------+---------+---------+-----------------+----------+----------+
|total_rows|        avg_sales|min_sales|max_sales|       avg_profit|min_profit|max_profit|
+----------+-----------------+---------+---------+-----------------+----------+----------+
|      9994|229.8580008304968|    0.444| 22638.48|28.65689630778464| -6599.978|  8399.976|
+----------+-----------------+---------+---------+-----------------+----------+----------+



In [17]:
#Grouping data
print("\n Sales summary by Category ")
category_summary = (df.groupBy("Category").agg(
          count("*").alias("order_count"),
          sum("Sales").alias("total_sales"),
          avg("Sales").alias("avg_sales"),
          sum("Profit").alias("total_profit"),)
      .orderBy(desc("total_sales")))
category_summary.show()

print("\n Sales summary by Region ")
region_summary = (
    df.groupBy("Region")
      .agg(count("*").alias("order_count"),sum("Sales").alias("total_sales"),
          avg("Sales").alias("avg_sales")).orderBy(desc("total_sales"),))
region_summary.show()

print("\n Sales summary by Region + Category ")
region_category_summary = (df.groupBy("Region", "Category").agg(sum("Sales").alias("total_sales"),sum("Profit").alias("total_profit"),)
      .orderBy("Region", desc("total_sales")))
region_category_summary.show(20)




 Sales summary by Category 
+---------------+-----------+-----------------+-----------------+------------------+
|       Category|order_count|      total_sales|        avg_sales|      total_profit|
+---------------+-----------+-----------------+-----------------+------------------+
|     Technology|       1847|836154.0329999996|452.7092761234432|145454.94810000004|
|      Furniture|       2121|741999.7953000009|349.8348869872706|        18451.2728|
|Office Supplies|       6026| 719047.032000001| 119.324100896117|122490.80079999991|
+---------------+-----------+-----------------+-----------------+------------------+


 Sales summary by Region 
+-------+-----------+-----------------+------------------+
| Region|order_count|      total_sales|         avg_sales|
+-------+-----------+-----------------+------------------+
|   West|       3203|725457.8245000001|226.49323275054638|
|   East|       2848|678781.2400000005|238.33610955056196|
|Central|       2323|501239.8907999995|215.7726606973

In [24]:
#  Build a Simple Pipeline (Load -> Clean -> Filter -> Transform -> Aggregate)


def run_pipeline():
    # Load
    raw = (spark.read.option("header", True).option("inferSchema", True)
        .option("quote", '"').option("escape", '"').option("multiLine", True)
        .csv("/content/Superstore.csv"))

    # Clean
    cleaned = raw.dropDuplicates()
    cleaned = cleaned.dropna()

    # Filter (example: keep only profitable orders)
    filtered = cleaned.filter(col("Profit") > 0)

    # Transform (rename + cast)
    transformed = (
        filtered.withColumnRenamed("Sub-Category", "sub_category")
                .withColumn("Sales", col("Sales").cast(DoubleType()))
                .withColumn("Profit", col("Profit").cast(DoubleType()))
    )

    # Aggregate
    result = (transformed.groupBy("Category", "sub_category").agg(count("*").alias("order_count"),sum("Sales").alias("total_sales"),
                       sum("Profit").alias("total_profit"),avg("Profit").alias("avg_profit"),)
                   .orderBy(desc("total_sales")))
    return result


print("\n Running pipeline ")
pipeline_result = run_pipeline()
pipeline_result.show(20)




 Running pipeline 
+---------------+------------+-----------+------------------+------------------+------------------+
|       Category|sub_category|order_count|       total_sales|      total_profit|        avg_profit|
+---------------+------------+-----------+------------------+------------------+------------------+
|     Technology|      Phones|        751|293993.89399999985|52046.354099999946| 69.30273515312909|
|      Furniture|      Chairs|        362|220870.14100000027| 36471.00760000003|100.74863977900561|
|Office Supplies|     Storage|        661|177559.33799999993|        27705.1302| 41.91396399394856|
|Office Supplies|     Binders|        910|167272.12000000008| 68732.25970000007| 75.52995571428579|
|     Technology| Accessories|        683|        156396.846|42867.262200000005| 62.76319502196194|
|     Technology|     Copiers|         68| 149528.0299999999|        55617.8249| 817.9091897058823|
|     Technology|    Machines|         71|        116782.378| 33503.42510000001|

In [20]:
#Saving result in csv
pipeline_result.toPandas().to_csv("results.csv", index=False)

print("Pipeline completed successfully!")

Pipeline completed successfully!
